In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

path = '/content/drive/MyDrive/Colab Notebooks/Prog/'
datafile = "Dataset1/output.txt"
max_word_len = 25
char_list = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','x','y','z','‘',"ʼ"]
char2idx = {c: i+1 for i, c in enumerate(char_list)}  # index from 1
char2idx['<PAD>'] = 0
idx2char = {v: k for k, v in char2idx.items()}
vocab_size = len(char2idx)

def encode_word(word):
    encoded = [char2idx.get(c, 0) for c in word]
    if len(encoded) < max_word_len:
        encoded += [0] * (max_word_len - len(encoded))
    return encoded[:max_word_len]

def decode_word(indices):
    return ''.join([idx2char.get(i, '') for i in indices if i != 0])

class StemDataset(Dataset):
    def __init__(self, filepath):
        self.data = []
        with open(filepath, 'r', encoding='utf8') as file:
            for line in file:
                line = line.strip()
                if '/' not in line: continue
                stem, affix = line.split('/')
                word = stem + affix
                self.data.append((encode_word(word), encode_word(stem)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

class BiLSTMStemmer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super(BiLSTMStemmer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        out = self.fc(x)
        return out

def train_model(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        outputs = outputs.permute(0, 2, 1)  # for CrossEntropy
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, device):
    model.eval()
    total = 0
    correct = 0
    from Levenshtein import distance as levenshtein_distance
    total_levenshtein = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=2).cpu().numpy()
            targets = targets.numpy()
            for pred_seq, tgt_seq in zip(predictions, targets):
                pred_word = decode_word(pred_seq)
                tgt_word = decode_word(tgt_seq)
                if pred_word == tgt_word:
                    correct += 1
                total_levenshtein += levenshtein_distance(pred_word, tgt_word)
                total += 1
    acc = 100 * correct / total
    avg_lev = total_levenshtein / total
    return acc, avg_lev

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = StemDataset(os.path.join(path, datafile))
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = BiLSTMStemmer(vocab_size=vocab_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        loss = train_model(model, dataloader, criterion, optimizer, device)
        acc, lev = evaluate_model(model, dataloader, device)
        print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Accuracy: {acc:.2f}% | Avg Levenshtein: {lev:.2f}")

import matplotlib.pyplot as plt
import datetime

def save_model(model, path):
    torch.save(model.state_dict(), path)

def save_errors(errors, path):
    with open(path, 'w', encoding='utf-8') as f:
        for tgt, pred in errors:
            f.write(f"{tgt}	{pred} \n")

def plot_metrics(accs, losses, output_dir):
    epochs = list(range(1, len(accs)+1))
    plt.figure()
    plt.plot(epochs, accs, label='Accuracy (%)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'accuracy_plot.png'))
    plt.close()

    plt.figure()
    plt.plot(epochs, losses, color='red', label='Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(os.path.join(output_dir, 'loss_plot.png'))
    plt.close()

if __name__ == '__main__':
    from sklearn.model_selection import train_test_split

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    full_dataset = StemDataset(os.path.join(path, datafile))
    indices = list(range(len(full_dataset)))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

    train_set = torch.utils.data.Subset(full_dataset, train_idx)
    test_set = torch.utils.data.Subset(full_dataset, test_idx)

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

    model = BiLSTMStemmer(vocab_size=vocab_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_acc = 0
    accs = []
    losses = []
    logs = []

    output_dir = os.path.join(path, 'BiLSTM_Results_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(output_dir, exist_ok=True)

    for epoch in range(10):
        loss = train_model(model, train_loader, criterion, optimizer, device)
        acc, lev = evaluate_model(model, test_loader, device)
        accs.append(acc)
        losses.append(loss)
        log_line = f"Epoch {epoch+1} | Loss: {loss:.4f} | Accuracy: {acc:.2f}% | Avg Levenshtein: {lev:.2f}"
        print(log_line)
        logs.append(log_line)

        if acc > best_acc:
            best_acc = acc
            save_model(model, os.path.join(output_dir, 'best_model.pt'))

    with open(os.path.join(output_dir, 'training_log.txt'), 'w') as f:
        for line in logs:
            f.write(line + '\n')

    # Save final weights and biases
    weights = model.fc.weight.detach().cpu().numpy()
    bias = model.fc.bias.detach().cpu().numpy()
    np.savetxt(os.path.join(output_dir, 'final_weights.txt'), weights)
    np.savetxt(os.path.join(output_dir, 'final_bias.txt'), bias)

    # Save error samples
    model.eval()
    errors = []
    from Levenshtein import distance as levenshtein_distance
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=2).cpu().numpy()
            targets = targets.numpy()
            for pred_seq, tgt_seq in zip(predictions, targets):
                pred_word = decode_word(pred_seq)
                tgt_word = decode_word(tgt_seq)
                if pred_word != tgt_word:
                    errors.append((tgt_word, pred_word))
    save_errors(errors, os.path.join(output_dir, 'error_samples.txt'))
    plot_metrics(accs, losses, output_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1 | Loss: 0.4425 | Accuracy: 63.81% | Avg Levenshtein: 0.60
Epoch 2 | Loss: 0.0571 | Accuracy: 74.95% | Avg Levenshtein: 0.42
Epoch 3 | Loss: 0.0435 | Accuracy: 79.93% | Avg Levenshtein: 0.33
Epoch 4 | Loss: 0.0370 | Accuracy: 82.51% | Avg Levenshtein: 0.28
Epoch 5 | Loss: 0.0321 | Accuracy: 83.93% | Avg Levenshtein: 0.25
Epoch 6 | Loss: 0.0284 | Accuracy: 84.04% | Avg Levenshtein: 0.24
Epoch 7 | Loss: 0.0262 | Accuracy: 87.16% | Avg Levenshtein: 0.20
Epoch 8 | Loss: 0.0238 | Accuracy: 88.34% | Avg Levenshtein: 0.17
Epoch 9 | Loss: 0.0215 | Accuracy: 85.36% | Avg Levenshtein: 0.21
Epoch 10 | Loss: 0.0198 | Accuracy: 89.46% | Avg Levenshtein: 0.14
Epoch 1 | Loss: 0.5333 | Accuracy: 58.72% | Avg Levenshtein: 0.72
Epoch 2 | Loss: 0.0643 | Accuracy: 71.91% | Avg Levenshtein: 0.48
Epoch 3 | Loss: 0.0462 | Accuracy: 76.17% | Avg Levenshtein: 0.40
Epoch 4 | Lo